<a href="https://colab.research.google.com/github/awaisfarooqchaudhry/IB9AU-GenerativeAI-2026/blob/main/Task16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 16 — Add a `get_pe_ratio` Tool to an Agent (Google Colab)

This notebook is a **complete beginner-friendly Colab notebook** for **Required Task 16**.

It is adapted from your uploaded **`Agents1.ipynb`**, but updated to use the current official **Google GenAI SDK** and a cleaner Colab workflow.

## What Task 16 asks you to do
You need to:
1. add a new tool called **`get_pe_ratio`**
2. fetch Apple's **Price-to-Earnings ratio**
3. ask the agent whether Apple is **"overvalued"** compared to an average market **P/E of 25**

## What this notebook does
- installs the required libraries
- configures the Gemini API
- defines three tools:
  - `get_stock_price`
  - `get_company_risk_score`
  - `get_pe_ratio`
- creates an agent with tool use enabled
- tests the tools
- asks the final Task 16 question:
  - **"Is Apple overvalued compared to the average market P/E of 25?"**
- shows the tool calls the agent made

## Why this notebook is slightly different from the lab notebook
Your uploaded `Agents1.ipynb` uses the older `google-generativeai` package.

Google's current official docs recommend using the **Google GenAI SDK** instead:
- Python package: `google-genai`

So this notebook keeps the same agent/tool idea, but uses the newer and recommended SDK.


## Step 0 — Install packages

This cell installs:
- `google-genai` for Gemini
- `yfinance` for stock data

It avoids upgrading core Colab packages unnecessarily.


In [1]:
!pip -q install google-genai yfinance

print("✅ Packages installed.")

✅ Packages installed.


## Step 1 — Add your Gemini API key

Paste your Gemini API key when asked.

You can create a key in Google AI Studio.


In [2]:
import os
from getpass import getpass

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Paste your Gemini API key: ")

if os.environ.get("GEMINI_API_KEY"):
    print("✅ Gemini API key loaded for this session.")
else:
    print("⚠️ No API key found.")

Paste your Gemini API key: ··········
✅ Gemini API key loaded for this session.


## Step 2 — Imports


In [3]:
import os
import json
from typing import Any

import yfinance as yf
from google import genai
from google.genai import types

## Step 3 — Create the Gemini client

We use **`gemini-2.5-flash`** because it is fast and works well for tool use.


In [4]:
MODEL_NAME = "gemini-2.5-flash"

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

print("✅ Gemini client ready.")
print("Model:", MODEL_NAME)

✅ Gemini client ready.
Model: gemini-2.5-flash


## Step 4 — Define the original tools from the lab

These match the spirit of your uploaded `Agents1.ipynb`.


In [5]:
def get_stock_price(ticker: str) -> dict:
    """Get the latest stock price for a ticker from Yahoo Finance.

    Args:
        ticker: Stock ticker symbol such as AAPL or MSFT.

    Returns:
        A dictionary with the ticker and latest price, or an error message.
    """
    print(f"🔍 TOOL CALL: get_stock_price({ticker})")

    try:
        stock = yf.Ticker(ticker)

        price = None
        try:
            fast_info = stock.fast_info
            price = fast_info.get("lastPrice", None) or fast_info.get("last_price", None)
        except Exception:
            price = None

        if price is None:
            info = stock.info
            price = info.get("currentPrice") or info.get("regularMarketPrice")

        if price is None:
            return {"ticker": ticker, "price": None, "error": "Price unavailable"}

        return {"ticker": ticker, "price": round(float(price), 2)}

    except Exception as e:
        return {"ticker": ticker, "price": None, "error": str(e)}


def get_company_risk_score(ticker: str) -> dict:
    """Estimate a simple company risk score using beta from Yahoo Finance.

    Args:
        ticker: Stock ticker symbol such as AAPL or NVDA.

    Returns:
        A dictionary containing the ticker, beta, and a simple risk assessment.
    """
    print(f"⚠️ TOOL CALL: get_company_risk_score({ticker})")

    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        beta = info.get("beta", None)

        if beta is None:
            return {
                "ticker": ticker,
                "beta": None,
                "assessment": "Risk data unavailable"
            }

        beta = float(beta)

        if beta > 1.5:
            assessment = "High Risk (High Volatility)"
        elif beta < 0.8:
            assessment = "Low Risk (Stable)"
        else:
            assessment = "Moderate Risk"

        return {
            "ticker": ticker,
            "beta": round(beta, 3),
            "assessment": assessment
        }

    except Exception as e:
        return {
            "ticker": ticker,
            "beta": None,
            "assessment": f"Risk data unavailable: {e}"
        }

print("✅ Base tools defined.")

✅ Base tools defined.


## Step 5 — Add the new Task 16 tool: `get_pe_ratio`

This is the key new tool required by the assignment.

We use the hint from the task:
```python
stock = yf.Ticker(ticker)
pe = stock.info.get('trailingPE', 'N/A')
```

I also add a small fallback to `forwardPE` if `trailingPE` is missing.


In [6]:
def get_pe_ratio(ticker: str) -> dict:
    """Get the Price-to-Earnings (P/E) ratio for a stock from Yahoo Finance.

    Args:
        ticker: Stock ticker symbol such as AAPL.

    Returns:
        A dictionary containing the ticker, trailing PE, and a short note.
    """
    print(f"💰 TOOL CALL: get_pe_ratio({ticker})")

    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        trailing_pe = info.get("trailingPE", None)
        forward_pe = info.get("forwardPE", None)

        if trailing_pe is not None:
            return {
                "ticker": ticker,
                "pe_ratio": round(float(trailing_pe), 3),
                "pe_type": "trailingPE",
                "note": "Trailing P/E fetched successfully"
            }

        if forward_pe is not None:
            return {
                "ticker": ticker,
                "pe_ratio": round(float(forward_pe), 3),
                "pe_type": "forwardPE",
                "note": "Trailing P/E unavailable, using forward P/E instead"
            }

        return {
            "ticker": ticker,
            "pe_ratio": None,
            "pe_type": None,
            "note": "P/E ratio unavailable from Yahoo Finance"
        }

    except Exception as e:
        return {
            "ticker": ticker,
            "pe_ratio": None,
            "pe_type": None,
            "note": f"Error fetching P/E ratio: {e}"
        }

print("✅ Task 16 tool defined.")

✅ Task 16 tool defined.


## Step 6 — Quick manual tool tests

Before giving the tools to the agent, we test them directly.


In [7]:
print(get_stock_price("AAPL"))
print(get_company_risk_score("AAPL"))
print(get_pe_ratio("AAPL"))

🔍 TOOL CALL: get_stock_price(AAPL)
{'ticker': 'AAPL', 'price': 251.49}
⚠️ TOOL CALL: get_company_risk_score(AAPL)
{'ticker': 'AAPL', 'beta': 1.116, 'assessment': 'Moderate Risk'}
💰 TOOL CALL: get_pe_ratio(AAPL)
{'ticker': 'AAPL', 'pe_ratio': 31.794, 'pe_type': 'trailingPE', 'note': 'Trailing P/E fetched successfully'}


## Step 7 — Create the agent configuration with automatic function calling

Google's current SDK can use **Python functions directly as tools**.
That means the SDK:
- turns them into tool declarations
- lets the model call them
- runs them automatically
- sends tool results back to the model
- returns the final answer


In [8]:
tools_list = [
    get_stock_price,
    get_company_risk_score,
    get_pe_ratio
]

agent_config = types.GenerateContentConfig(
    tools=tools_list
)

print("✅ Agent config created with 3 tools.")

✅ Agent config created with 3 tools.


## Step 8 — Test a simple tool-use question

This makes sure the model can actually call the tools.


In [9]:
simple_query = "What is Apple's current stock price and what is its P/E ratio?"
simple_response = client.models.generate_content(
    model=MODEL_NAME,
    contents=simple_query,
    config=agent_config
)

print("🤖 AGENT RESPONSE:")
print(simple_response.text)

🔍 TOOL CALL: get_stock_price(AAPL)
💰 TOOL CALL: get_pe_ratio(AAPL)
🤖 AGENT RESPONSE:
Apple's current stock price is $251.49 and its P/E ratio is 31.794.


## Step 9 — Required Task 16 question

Now we ask the actual assignment question.

We give the model the comparison threshold directly:
- average market P/E = **25**

The model should use `get_pe_ratio("AAPL")` and then decide whether Apple looks overvalued relative to 25.


In [10]:
task16_query = """
Use the available tools to answer this question.

Question:
Is Apple overvalued compared to the average market P/E of 25?

Instructions:
- First get Apple's P/E ratio using the appropriate tool.
- Then compare it with the average market P/E of 25.
- If Apple's P/E is materially above 25, explain that it appears overvalued on this simple P/E comparison.
- If it is below or close to 25, explain that it does not appear overvalued on this simple P/E comparison.
- Keep the answer concise and financially clear.
""".strip()

task16_response = client.models.generate_content(
    model=MODEL_NAME,
    contents=task16_query,
    config=agent_config
)

print("🤖 TASK 16 AGENT RESPONSE:")
print(task16_response.text)

💰 TOOL CALL: get_pe_ratio(AAPL)
🤖 TASK 16 AGENT RESPONSE:
Apple's P/E ratio is 31.794. Compared to the average market P/E of 25, Apple appears overvalued on this simple P/E comparison.


## Step 10 — Inspect the tool calls the model made

This is useful because your lecturer can clearly see that the model used the new `get_pe_ratio` tool.


In [11]:
print("===== TOOL CALL INSPECTION =====")

parts = task16_response.candidates[0].content.parts
found_tool_call = False

for i, part in enumerate(parts):
    if getattr(part, "function_call", None):
        found_tool_call = True
        fn = part.function_call
        print(f"Tool call #{i+1}:")
        print("  name :", fn.name)
        print("  args :", dict(fn.args))
        print()

if not found_tool_call:
    print("No function call was visible in the final response object.")
    print("That can happen because the SDK already handled the tool call automatically before returning final text.")

===== TOOL CALL INSPECTION =====
No function call was visible in the final response object.
That can happen because the SDK already handled the tool call automatically before returning final text.


## Step 11 — Manual verification cell

This cell does not use the agent.
It computes the simple overvalued / not-overvalued judgment directly from the tool output.

This gives you a transparent audit check.


In [12]:
pe_result = get_pe_ratio("AAPL")
market_pe = 25

print("Apple PE result:", pe_result)
print("Average market PE threshold:", market_pe)

if pe_result["pe_ratio"] is None:
    print("Could not verify because Apple's P/E ratio was unavailable.")
else:
    if pe_result["pe_ratio"] > market_pe:
        print("Manual check: Apple appears overvalued on this simple P/E comparison.")
    else:
        print("Manual check: Apple does not appear overvalued on this simple P/E comparison.")

💰 TOOL CALL: get_pe_ratio(AAPL)
Apple PE result: {'ticker': 'AAPL', 'pe_ratio': 31.794, 'pe_type': 'trailingPE', 'note': 'Trailing P/E fetched successfully'}
Average market PE threshold: 25
Manual check: Apple appears overvalued on this simple P/E comparison.


## Step 12 — Save the result to a text file

This is convenient if you want to submit or download the final answer.


In [13]:
from pathlib import Path

Path("task16_outputs").mkdir(exist_ok=True)

with open("task16_outputs/task16_agent_answer.txt", "w", encoding="utf-8") as f:
    f.write("Task 16 Query:\n")
    f.write(task16_query + "\n\n")
    f.write("Task 16 Agent Response:\n")
    f.write(task16_response.text + "\n\n")
    f.write("Manual Verification:\n")
    f.write(json.dumps(pe_result, indent=2))
    f.write("\n")
    if pe_result["pe_ratio"] is not None:
        verdict = (
            "Apple appears overvalued on this simple P/E comparison."
            if pe_result["pe_ratio"] > market_pe
            else "Apple does not appear overvalued on this simple P/E comparison."
        )
        f.write(verdict + "\n")

print("✅ Saved task16_outputs/task16_agent_answer.txt")

✅ Saved task16_outputs/task16_agent_answer.txt


## Step 13 — Download the output


In [14]:
from google.colab import files
files.download("task16_outputs/task16_agent_answer.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Troubleshooting

### If you get a Gemini authentication error
Check that:
- your API key was pasted correctly
- your API key is active
- billing / quota is available if required

### If Yahoo Finance returns missing data
That can happen sometimes.
Possible reasons:
- temporary Yahoo response issue
- `trailingPE` not available
- ticker metadata not fully populated

The notebook already falls back to `forwardPE` if `trailingPE` is missing.

### If the model answers without clearly showing tool calls
That is normal with automatic function calling.
The SDK may already have executed the tools and returned only the final answer.

### If you want an even stricter financial framing
You can change the query to say:
"Judge overvalued only on the basis of simple P/E comparison, not on cash flow, growth, or margins."
